In [1]:
# ======================================
# 🧠 Resume Training for GTSRB Model
# ======================================

import os
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import load_model
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from sklearn.model_selection import train_test_split
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.preprocessing import image
from tqdm import tqdm



In [2]:
# -------------------------------
# Paths and Constants
# -------------------------------
DATA_DIR = "./data"
TRAIN_CSV = os.path.join(DATA_DIR, "Train.csv")
TEST_CSV = os.path.join(DATA_DIR, "Test.csv")
MODEL_PATH = "./models/traffic_sign_detection_gtsrb.h5"
IMG_SIZE = (32, 32)
BATCH_SIZE = 64
EPOCHS = 12
NUM_CLASSES = 43

# -------------------------------
# Load CSV Data
# -------------------------------
train_df = pd.read_csv(TRAIN_CSV)
test_df = pd.read_csv(TEST_CSV)

print(f"✅ Train samples: {len(train_df)}, Test samples: {len(test_df)}")

# -------------------------------
# Preprocessing Function
# -------------------------------
def preprocess_images(df, base_dir):
    images = []
    labels = []

    for i in tqdm(range(df.shape[0]), desc=f"Loading images from {base_dir}"):
        img_path = os.path.join(base_dir, df['Path'].iloc[i])
        label = df['ClassId'].iloc[i]
        try:
            img = image.load_img(img_path, target_size=IMG_SIZE)
            img = image.img_to_array(img)
            img = img / 255.0
            images.append(img)
            labels.append(label)
        except Exception as e:
            print(f"⚠️ Error loading image {img_path}: {e}")

    X = np.array(images)
    y = to_categorical(np.array(labels), NUM_CLASSES)
    return X, y

# -------------------------------
# Load Train and Validation Data
# -------------------------------
X_train_full, y_train_full = preprocess_images(train_df, os.path.join(DATA_DIR))
X_train, X_val, y_train, y_val = train_test_split(X_train_full, y_train_full, test_size=0.2, random_state=42)

print(f"Train shape: {X_train.shape}, Validation shape: {X_val.shape}")



✅ Train samples: 39209, Test samples: 12630


Loading images from ./data: 100%|██████████| 39209/39209 [00:35<00:00, 1099.02it/s]


Train shape: (31367, 32, 32, 3), Validation shape: (7842, 32, 32, 3)


In [3]:
# -------------------------------
# Load Existing Model
# -------------------------------
if os.path.exists(MODEL_PATH):
    print("✅ Loading saved model...")
    model = load_model(MODEL_PATH)
else:
    raise FileNotFoundError(f"❌ Saved model not found at {MODEL_PATH}")

# -------------------------------
# Data Augmentation
# -------------------------------
datagen = ImageDataGenerator(
    rotation_range=10,
    zoom_range=0.1,
    width_shift_range=0.1,
    height_shift_range=0.1
)

datagen.fit(X_train)

# -------------------------------
# Callbacks
# -------------------------------
checkpoint_path = "./models/traffic_sign_detection_gtsrb_resume.h5"

callbacks = [
    EarlyStopping(monitor='val_loss', patience=4, restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.3, patience=2, min_lr=1e-6),
    ModelCheckpoint(checkpoint_path, monitor='val_accuracy', save_best_only=True, verbose=1)
]



✅ Loading saved model...


d:\CGC\PHD\conf\paper\traffic_sign_detection_gtsrb\.venv\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [5]:
# -------------------------------
# Resume Training
# -------------------------------
model.compile(
    optimizer=tf.keras.optimizers.Adam(),   # new optimizer instance
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

history = model.fit(
    datagen.flow(X_train, y_train, batch_size=BATCH_SIZE),
    validation_data=(X_val, y_val),
    epochs=EPOCHS,
    callbacks=callbacks,
    verbose=1
)

# -------------------------------
# Save Updated Model
# -------------------------------
model.save("./models/traffic_sign_detection_gtsrb_updated.h5")
print("✅ Training resumed and updated model saved at './models/traffic_sign_detection_gtsrb_updated.h5'")



Epoch 1/12
491/491 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step - accuracy: 0.7474 - loss: 1.1031
Epoch 1: val_accuracy improved from None to 0.99554, saving model to ./models/traffic_sign_detection_gtsrb_resume.h5


491/491 ━━━━━━━━━━━━━━━━━━━━ 29s 55ms/step - accuracy: 0.8019 - loss: 0.7691 - val_accuracy: 0.9955 - val_loss: 0.0198 - learning_rate: 0.0010
Epoch 2/12
491/491 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step - accuracy: 0.8767 - loss: 0.4350
Epoch 2: val_accuracy did not improve from 0.99554
491/491 ━━━━━━━━━━━━━━━━━━━━ 25s 51ms/step - accuracy: 0.8899 - loss: 0.3854 - val_accuracy: 0.9954 - val_loss: 0.0166 - learning_rate: 0.0010
Epoch 3/12
491/491 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step - accuracy: 0.9114 - loss: 0.3014
Epoch 3: val_accuracy did not improve from 0.99554
491/491 ━━━━━━━━━━━━━━━━━━━━ 29s 60ms/step - accuracy: 0.9168 - loss: 0.2836 - val_accuracy: 0.9954 - val_loss: 0.0151 - learning_rate: 0.0010
Epoch 4/12
491/491 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step - accuracy: 0.9284 - loss: 0.2449
Epoch 4: val_accuracy did not improve from 0.99554
491/491 ━━━━━━━━━━━━━━━━━━━━ 27s 55ms/step - accuracy: 0.9305 - loss: 0.2347 - val_accuracy: 0.9949 - val_loss: 0.0152 - learning_rate: 0.0010
Epoch 5/12
491

491/491 ━━━━━━━━━━━━━━━━━━━━ 24s 50ms/step - accuracy: 0.9482 - loss: 0.1732 - val_accuracy: 0.9958 - val_loss: 0.0148 - learning_rate: 0.0010
Epoch 7/12
491/491 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step - accuracy: 0.9550 - loss: 0.1556
Epoch 7: val_accuracy improved from 0.99579 to 0.99617, saving model to ./models/traffic_sign_detection_gtsrb_resume.h5


491/491 ━━━━━━━━━━━━━━━━━━━━ 27s 55ms/step - accuracy: 0.9566 - loss: 0.1540 - val_accuracy: 0.9962 - val_loss: 0.0129 - learning_rate: 0.0010
Epoch 8/12
491/491 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step - accuracy: 0.9573 - loss: 0.1428
Epoch 8: val_accuracy did not improve from 0.99617
491/491 ━━━━━━━━━━━━━━━━━━━━ 26s 54ms/step - accuracy: 0.9592 - loss: 0.1371 - val_accuracy: 0.9949 - val_loss: 0.0143 - learning_rate: 0.0010
Epoch 9/12
491/491 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step - accuracy: 0.9589 - loss: 0.1379
Epoch 9: val_accuracy did not improve from 0.99617
491/491 ━━━━━━━━━━━━━━━━━━━━ 24s 48ms/step - accuracy: 0.9603 - loss: 0.1324 - val_accuracy: 0.9958 - val_loss: 0.0147 - learning_rate: 0.0010
Epoch 10/12
490/491 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step - accuracy: 0.9671 - loss: 0.1089
Epoch 10: val_accuracy improved from 0.99617 to 0.99821, saving model to ./models/traffic_sign_detection_gtsrb_resume.h5


491/491 ━━━━━━━━━━━━━━━━━━━━ 26s 52ms/step - accuracy: 0.9688 - loss: 0.1028 - val_accuracy: 0.9982 - val_loss: 0.0074 - learning_rate: 3.0000e-04
Epoch 11/12
491/491 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step - accuracy: 0.9707 - loss: 0.1000
Epoch 11: val_accuracy did not improve from 0.99821
491/491 ━━━━━━━━━━━━━━━━━━━━ 26s 53ms/step - accuracy: 0.9728 - loss: 0.0935 - val_accuracy: 0.9980 - val_loss: 0.0076 - learning_rate: 3.0000e-04
Epoch 12/12
490/491 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step - accuracy: 0.9739 - loss: 0.0864
Epoch 12: val_accuracy did not improve from 0.99821
491/491 ━━━━━━━━━━━━━━━━━━━━ 26s 54ms/step - accuracy: 0.9736 - loss: 0.0890 - val_accuracy: 0.9974 - val_loss: 0.0090 - learning_rate: 3.0000e-04


✅ Training resumed and updated model saved at './models/traffic_sign_detection_gtsrb_updated.h5'


In [6]:
# -------------------------------
# Evaluate on Test Data
# -------------------------------
X_test, y_test = preprocess_images(test_df, os.path.join(DATA_DIR))
test_loss, test_acc = model.evaluate(X_test, y_test, verbose=1)
print(f"🎯 Test Accuracy: {test_acc*100:.2f}%")


Loading images from ./data: 100%|██████████| 12630/12630 [00:19<00:00, 651.24it/s]


395/395 ━━━━━━━━━━━━━━━━━━━━ 3s 8ms/step - accuracy: 0.9770 - loss: 0.1045
🎯 Test Accuracy: 97.70%


In [7]:
history = model.fit(
    datagen.flow(X_train, y_train, batch_size=BATCH_SIZE),
    validation_data=(X_val, y_val),
    epochs=EPOCHS,
    callbacks=callbacks,
    verbose=1
)

# -------------------------------
# Save Updated Model
# -------------------------------
model.save("./models/Resnet50_taffic_sign_detection_gtsrb_updatedV2.h5")
print("✅ Training resumed and updated model saved at './models/traffic_sign_detection_gtsrb_updated.h5'")



Epoch 1/12
491/491 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - accuracy: 0.9752 - loss: 0.0870
Epoch 1: val_accuracy did not improve from 0.99821
491/491 ━━━━━━━━━━━━━━━━━━━━ 14s 28ms/step - accuracy: 0.9746 - loss: 0.0890 - val_accuracy: 0.9978 - val_loss: 0.0069 - learning_rate: 9.0000e-05
Epoch 2/12
491/491 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step - accuracy: 0.9749 - loss: 0.0816
Epoch 2: val_accuracy did not improve from 0.99821
491/491 ━━━━━━━━━━━━━━━━━━━━ 15s 31ms/step - accuracy: 0.9750 - loss: 0.0825 - val_accuracy: 0.9982 - val_loss: 0.0072 - learning_rate: 9.0000e-05
Epoch 3/12
490/491 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step - accuracy: 0.9750 - loss: 0.0797
Epoch 3: val_accuracy did not improve from 0.99821
491/491 ━━━━━━━━━━━━━━━━━━━━ 23s 47ms/step - accuracy: 0.9750 - loss: 0.0802 - val_accuracy: 0.9978 - val_loss: 0.0073 - learning_rate: 9.0000e-05
Epoch 4/12
490/491 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step - accuracy: 0.9785 - loss: 0.0769
Epoch 4: val_accuracy did not improve from 0.99821
491/491

✅ Training resumed and updated model saved at './models/traffic_sign_detection_gtsrb_updated.h5'


In [8]:
# -------------------------------
# Evaluate on Test Data
# -------------------------------
# X_test, y_test = preprocess_images(test_df, os.path.join(DATA_DIR))
test_loss, test_acc = model.evaluate(X_test, y_test, verbose=1)
print(f"🎯 Test Accuracy: {test_acc*100:.2f}%")


395/395 ━━━━━━━━━━━━━━━━━━━━ 3s 8ms/step - accuracy: 0.9806 - loss: 0.0922
🎯 Test Accuracy: 98.06%
